# Week 2 Day 3 — Reproducibility: Seed, Logs, Metrics
**Jul 10, 2026**

Closing out the week: making a training run **repeatable** (same seed → same result, byte for byte) and **inspectable after the fact** (metrics saved to a file, not just printed and lost when the kernel closes). Both matter for the same underlying reason — without them, you can't tell whether a change you made actually helped, or whether you're just looking at noise.

Scaffold as usual: TODO stubs, hints not formulas, self-check cells.

## Part 1: `set_seed`

TODO: write `set_seed(seed)` that makes a training run deterministic.

You've only ever called `torch.manual_seed` in this curriculum, but randomness in a real pipeline doesn't only come from `torch`. Three separate generators typically need seeding:

- Python's own `random` module (e.g. if you ever shuffle a list of file paths with it)
- `numpy`'s random state (e.g. `np.random.randn`, if data generation uses numpy anywhere)
- `torch`'s random state (covers `torch.randn`, weight initialization, `DataLoader` shuffling, dropout, etc.)

All three need to be set together — seeding only `torch` while a numpy-based step elsewhere stays unseeded would leave you with a run that's *partially* reproducible, which is a confusing state to debug.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

def set_seed(seed):
    # TODO: seed random, numpy, and torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Part 2: Prove it's actually reproducible

Given, once Part 1 is filled in — a full data-generation + training run wrapped in a function, called twice with the same seed. If `set_seed` is correct, the two runs should produce **identical** final losses and weights, not just similar ones.

In [2]:
def run_training(seed):
    set_seed(seed)

    n_samples, n_features = 200, 4
    X = torch.randn(n_samples, n_features)
    true_w = torch.tensor([0.8, -1.2, 0.5, 1.0])
    y = (X @ true_w > 0).float()

    model = nn.Sequential(nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, 1))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

    for epoch in range(10):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb).squeeze(-1), yb)
            loss.backward()
            optimizer.step()

    return loss.item(), list(model.parameters())[0].detach().clone()

loss1, w1 = run_training(seed=42)
loss2, w2 = run_training(seed=42)

assert loss1 == loss2, f"losses differ: {loss1} vs {loss2}"
assert torch.equal(w1, w2), "weights differ between runs"
print("reproducible: identical loss and weights across two runs with the same seed")

loss3, _ = run_training(seed=1)
print(f"seed=42 loss: {loss1:.6f}   seed=1 loss: {loss3:.6f}  (expected to differ)")

reproducible: identical loss and weights across two runs with the same seed
seed=42 loss: 0.103055   seed=1 loss: 0.061971  (expected to differ)


## Part 3: A metrics logger

TODO: write a small `MetricsLogger` class that collects per-epoch metrics during training and can write them to disk when you're done.

Minimum shape: a `.log(epoch, **metrics)` method that stores a record per call (e.g. append a dict like `{"epoch": epoch, "train_loss": ..., "val_loss": ...}` to an internal list), and a `.to_csv(path)` method that writes the accumulated records out — `pandas.DataFrame(records).to_csv(path, index=False)` is the easy way to turn a list of same-shaped dicts into a CSV in one line, if you want to lean on pandas rather than the `csv` module directly.

In [4]:
import pandas as pd
class MetricsLogger:
    def __init__(self):
        self.rows = []

    def log(self, epoch, **metrics):
        # TODO: record epoch + whatever metric kwargs were passed
        row = {"epoch": epoch, **metrics}
        self.rows.append(row)

    def to_csv(self, path):
        # TODO: write all recorded rows to `path`
        df = pd.DataFrame(self.rows)
        df.to_csv(path, index=False)

# self-check
logger = MetricsLogger()
logger.log(epoch=0, train_loss=0.9, val_loss=0.95)
logger.log(epoch=1, train_loss=0.7, val_loss=0.80)
logger.to_csv("metrics_test.csv")

import pandas as pd
check = pd.read_csv("metrics_test.csv")
assert len(check) == 2 and list(check["epoch"]) == [0, 1]
print(check)

   epoch  train_loss  val_loss
0      0         0.9      0.95
1      1         0.7      0.80


## Part 4: Put it together — seeded, logged training run

TODO: rerun training (reuse the setup from Part 2), but this time call `logger.log(...)` with train and val loss at the end of every epoch, then save the log to `day10_metrics.csv` when training finishes.

In [5]:
set_seed(42)

n_samples, n_features = 200, 4
X = torch.randn(n_samples, n_features)
true_w = torch.tensor([0.8, -1.2, 0.5, 1.0])
y = (X @ true_w > 0).float()

ds = TensorDataset(X, y)
gen = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(ds, [0.8, 0.2], generator=gen)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

model = nn.Sequential(nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.BCEWithLogitsLoss()
run_logger = MetricsLogger()

for epoch in range(20):
    model.train()
    train_loss_total, n_train = 0.0, 0
    for xb, yb in train_loader:
        # TODO: standard training step, accumulate train_loss_total (weighted by batch size, like Day 5 / Week 2 Day 2)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs.squeeze(), yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    model.eval()
    val_loss_total, n_val = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            # TODO: accumulate val_loss_total the same way
            loss = criterion(model(xb).squeeze(), yb)
            val_loss_total += loss.item() * yb.size(0)
            
            n_val += yb.size(0)

    # TODO: run_logger.log(epoch=epoch, train_loss=..., val_loss=...)
    train_loss = train_loss_total / n_train if n_train > 0 else 0.0
    val_loss = val_loss_total / n_val if n_val > 0 else 0
    run_logger.log(epoch=epoch, train_loss=train_loss, val_loss=val_loss)

run_logger.to_csv("metrics.csv")
print(pd.read_csv("metrics.csv"))

    epoch  train_loss  val_loss
0       0         0.0  0.617975
1       1         0.0  0.554710
2       2         0.0  0.492656
3       3         0.0  0.430814
4       4         0.0  0.370063
5       5         0.0  0.313056
6       6         0.0  0.264200
7       7         0.0  0.223409
8       8         0.0  0.189309
9       9         0.0  0.164589
10     10         0.0  0.143248
11     11         0.0  0.124539
12     12         0.0  0.108580
13     13         0.0  0.098593
14     14         0.0  0.091344
15     15         0.0  0.085830
16     16         0.0  0.081212
17     17         0.0  0.077836
18     18         0.0  0.073878
19     19         0.0  0.071384


## Try yourself

1. Load `metrics.csv` back with pandas and plot `train_loss`/`val_loss` vs. `epoch` with matplotlib — a training curve reconstructed purely from the logged file, no need to rerun training.
2. Add a `run_id` (e.g. a timestamp string) to every logged row, so multiple training runs can be appended to the same CSV and later filtered apart.
3. Look up `torch.backends.cudnn.deterministic` and `torch.backends.cudnn.benchmark` — why don't they matter on this CPU-only setup, and what would change on a GPU machine?
4. Log the current `seed` itself as a column in every row — so six months from now, a metrics file alone is enough to know exactly how to reproduce that run.